# 02 - LST pipeline (Phase 2)

**Colombo UHI practicum.** Builds the temperature layer every later phase reads:

1. a **harmonised Landsat 5/7/8/9 Collection-2 Level-2 collection** - QA_PIXEL
   bits 0-4 + `QA_RADSAT == 0` masking, per-sensor scale factors, and surface
   reflectance renamed to `blue`...`swir2` so index code is sensor-agnostic;
2. **annual and dry-season (Jan-Mar) composites**, each shipping a **per-pixel
   valid-observation count**;
3. **MODIS MOD11A2 / MYD11A2** day and night LST with explicit `QC_Day` /
   `QC_Night` bit filtering (good quality AND average error <= 1 K);
4. a **Landsat-vs-MODIS annual mean comparison** over the CMC.

Run top-to-bottom in **Google Colab** after `01_aoi_and_boundaries.ipynb` has
worked once. All logic lives in `src/colombo_uhi/`; this notebook orchestrates
and displays.

> **Caveat (CLAUDE.md #1):** every number here is **LAND SURFACE TEMPERATURE**,
> never air temperature. Surface UHI can be roughly 2x the canopy-air UHI.
>
> **Caveat (CLAUDE.md #2):** never read a composite without its `obs_count`
> band. Tropical cloud cover means only a minority of scenes are usable.
>
> **Caveat (CLAUDE.md #4):** Landsat sees one ~10:30 local overpass. Night-time
> UHI comes only from MODIS.

### If Earth Engine says "User memory limit exceeded"

It is almost always **graph depth** or **region size**, not pixel count. Levers,
in order of effect - all four are already applied below, so this list is for
when you extend the notebook:

1. **Drop the percentile band** (`with_percentile=False`) when you only need the
   mean. A percentile reducer retains every observation per pixel to sort them;
   mean and count are streaming accumulators.
2. **Batch long series** (`BATCH_YEARS`, Step 6). One request covering 26 annual
   composites holds 26 image graphs at once.
3. **Composite one year at a time** (`start_year == end_year`) instead of
   building the full series and filtering down to one year.
4. **Shrink `WORK_REGION`** (Step 1) and raise `ZONAL_SCALE_M` (Step 6).

None of these change any number - they change how the work is packaged. Changing
`ZONAL_SCALE_M` *does* change the number slightly, so state it when you report.

In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

# Which revision is actually on disk. Quote this if a result looks impossible.
!git --no-pager log -1 --format="HEAD %h %s (%ci)"

In [ ]:
# COLAB: RUN THIS CELL  (skip if you already ran notebook 00 or 01 in this runtime)
%pip install -q -r requirements.txt
print("\nIf Colab asked to RESTART the runtime: Runtime > Restart session,")
print("then re-run this notebook FROM THE CLONE CELL (skip this pip cell).")

In [ ]:
# COLAB: RUN THIS CELL
# Load params (single source of truth) and initialise Earth Engine.
import sys

sys.path.insert(0, os.path.abspath("src"))

# Drop any already-imported colombo_uhi modules BEFORE importing. Without this,
# re-running the notebook in a live runtime keeps the version cached in
# sys.modules from the previous run: `git pull` updates the files on disk but the
# import silently returns the OLD code, so new functions appear to not exist
# (AttributeError) and fixed bugs appear unfixed. This cost a full run in Phase 1d.
for _name in [m for m in list(sys.modules) if m == "colombo_uhi" or m.startswith("colombo_uhi.")]:
    del sys.modules[_name]

from colombo_uhi import load_params
from colombo_uhi.auth import init_ee

params = load_params()
project = init_ee()
print("Earth Engine initialised with project:", project)
print()
for _key in ("lst_not_air_temp", "valid_obs_required", "single_overpass"):
    print("CAVEAT:", " ".join(params["caveats"][_key].split()))
    print()

## Step 1 - geometries and the water mask, scoped tight

Everything downstream is built over `WORK_REGION` = **Colombo District**, not
the province-plus-25 km `analysis_region` that Phase 1 used for the rural ring.
The reflectance composite behind `aoi.water_mask` is built over whatever region
it is given, so this one choice dominates the memory cost of the whole notebook.

**Water is masked before any statistic** (CLAUDE.md). That matters more than it
sounds for Colombo: COD-AB's CMC polygon encloses ~6.9 km2 of Port outer
harbour, and leaving it in would drag the urban mean down.

In [ ]:
# COLAB: RUN THIS CELL
import ee

from colombo_uhi import aoi, composites, indices, landsat, modis, viz

# Fail fast and legibly if a STALE colombo_uhi is loaded (see the purge above).
_required = {
    "landsat": ["harmonised_collection", "monsoon_season", "output_band_names"],
    "composites": ["annual_composites", "dry_season_composites", "scene_inventory"],
    "modis": ["lst_collection", "annual_lst", "clear_sky_count"],
    "indices": ["add_indices", "albedo"],
    "viz": ["plot_annual_lst_comparison"],
}
_absent = {
    name: [f for f in funcs if not hasattr(globals()[name], f)]
    for name, funcs in _required.items()
}
_absent = {k: v for k, v in _absent.items() if v}
if _absent:
    raise RuntimeError(
        f"colombo_uhi modules are STALE, missing {_absent}.\n"
        f"  landsat loaded from: {landsat.__file__}\n"
        "Fix, in order:\n"
        "  1. Runtime > Restart session, then re-run from the CLONE cell.\n"
        "  2. If it persists, your local commits are not pushed - check that the\n"
        "     HEAD line printed by the clone cell is the revision you expect."
    )

district_fc = aoi.colombo_district(params)
district_geom = district_fc.geometry(10)

# Degrade gracefully if the CMC cannot be built, so the rest still runs
# (same pattern as notebook 01).
try:
    cmc_geom = aoi.cmc_boundary(params)
    print("CMC boundary built.")
except Exception as exc:  # noqa: BLE001 - surface the real reason, keep going
    cmc_geom = None
    print("COULD NOT BUILD THE CMC:", exc)
    print("Falling back to Colombo District for the zonal statistics below.")

zone_geom = cmc_geom if cmc_geom is not None else district_geom
zone_label = "CMC" if cmc_geom is not None else "Colombo District"

# THE memory knob for this notebook. Shrink it first if EE runs out of memory.
WORK_REGION = district_geom.bounds(10)

print("Zonal statistics will use:", zone_label)
print("Working region area (km2):", aoi.area_km2(WORK_REGION).getInfo())

In [ ]:
# COLAB: RUN THIS CELL
# Build the water mask ONCE, over the working region only. It composites
# Landsat internally, so both the scope and the reuse matter.
water = aoi.water_mask(params, region=WORK_REGION)
land = water.Not()
print("Water mask built over the working region.")
print("Thresholds:", {k: v for k, v in params["aoi"]["water_mask"].items() if k != "composite"})

## Step 2 - the harmonised Landsat collection

Four collections merged into one, every scene carrying `LST_C` (degrees Celsius)
plus `blue`/`green`/`red`/`nir`/`swir1`/`swir2` and `ST_QA_K`.

The diagnostic prints scene counts **with and without** the `PROCESSING_LEVEL`
filter, which drops `L2SR` scenes (surface reflectance only, thermal fully
masked). It is implemented as `neq("L2SR")` rather than `eq("L2SP")` precisely
so a renamed property cannot silently empty the collection - but check the drop
is small anyway.

In [ ]:
# COLAB: RUN THIS CELL
scenes = landsat.harmonised_collection(params, region=WORK_REGION)

# Band names come from the client-side helper, NOT from bandNames().getInfo():
# asking Earth Engine for them would force it to evaluate the whole image graph.
print("Harmonised band schema:", landsat.output_band_names(params))
print("Total scenes over the working region:", scenes.size().getInfo())
print()

_c2l2 = params["landsat_c2l2"]
_start = f"{params['time']['start_year']}-01-01"
_end = f"{params['time']['end_year'] + 1}-01-01"
print(f"{'sensor':<12}{'unfiltered':>12}{'L2SP only':>12}")
for _key in landsat.sensor_keys(params):
    _raw = (
        ee.ImageCollection(params["datasets"][_key]["id"])
        .filterBounds(WORK_REGION)
        .filterDate(_start, _end)
    )
    _kept = _raw.filter(
        ee.Filter.neq(_c2l2["processing_level_property"], _c2l2["processing_level_exclude"])
    )
    print(f"{_key:<12}{_raw.size().getInfo():>12}{_kept.size().getInfo():>12}")
print()
print("If an 'L2SP only' count is 0 while 'unfiltered' is not, the property name")
print("has changed: set landsat_c2l2.processing_level_filter_enabled: false.")

## Step 3 - scenes per year x sensor: where the data gaps are

Expect these gaps, and treat anything else as suspicious:

| Period | Expectation |
|---|---|
| 2000-2003 | Landsat 5 + Landsat 7, both SLC-on - the best early coverage |
| from 2003-06 | Landsat 7 striped (SLC-off); kept by default, median compositing dilutes it |
| 2012-05 to 2013-03 | **Landsat 7 only** - L5 ended 2012-05-05, L8 launched 2013-03-18 |
| from 2013-03 | Landsat 8 |
| from 2021-10 | Landsat 9 joins; L7 collection ends 2024-01 |

These are metadata-only reductions - one Earth Engine round trip each, no pixels.

In [ ]:
# COLAB: RUN THIS CELL
dry_months = params["time"]["seasons"]["dry_window"]["months"]
dry_scenes = landsat.harmonised_collection(params, region=WORK_REGION, months=dry_months)

inventory_all = composites.scene_inventory(scenes, params)
inventory_dry = composites.scene_inventory(dry_scenes, params)

print("SCENES PER YEAR x SENSOR - full year")
print(inventory_all.to_string())
print()
print(f"SCENES PER YEAR x SENSOR - dry window (months {dry_months})")
print(inventory_dry.to_string())
print()
_empty = inventory_dry.index[inventory_dry["total"] == 0].tolist()
print("Dry-window years with ZERO scenes:", _empty if _empty else "none")

## Step 4 - dry-season composite for a recent year

`dry_season_composites` restricts to Jan-Mar (the driest, clearest window) and
returns one image per year with three bands: `LST_C` (median), `LST_C_p90`, and
`obs_count`.

Only the mapped year is built here (`start_year == end_year`). Building all 26
and filtering down to one would force Earth Engine to evaluate every year's
graph just to read the `year` property off each - that is what exhausted the
memory limit on the first run.

In [ ]:
# COLAB: RUN THIS CELL
MAP_YEAR = 2025  # most recent complete Jan-Mar window inside the study period

recent = ee.Image(
    composites.dry_season_composites(
        scenes, params, start_year=MAP_YEAR, end_year=MAP_YEAR
    ).first()
).updateMask(land)

# Band names client-side; only metadata is fetched, in ONE round trip.
_expected_bands = composites.composite_band_names(
    params["landsat_c2l2"]["lst_band_name"], params["composites"]["percentile"], params
)
_meta = ee.Dictionary(
    {
        "n_scenes": recent.get(params["composites"]["n_scenes_property"]),
        "reducer": recent.get("reducer"),
        "percentile": recent.get("percentile"),
        "year": recent.get(params["composites"]["year_property"]),
    }
).getInfo()

print(f"Dry-season composite {MAP_YEAR}")
print("  bands       :", _expected_bands)
print("  metadata    :", _meta)

In [ ]:
# COLAB: RUN THIS CELL
# Static PNGs into figures/ - the interactive map renders nothing once the
# notebook is saved, so these are the reviewable evidence.
from IPython.display import Image, display

# Display-only stretches (not analysis constants).
LST_MIN_C, LST_MAX_C = 22, 42
OBS_MAX = 12
THERMAL = ["2c7bb6", "abd9e9", "ffffbf", "fdae61", "d7191c"]
COUNTS = ["440154", "31688e", "35b779", "fde725"]

outlines = [viz.outline_image(district_fc, "000000", 2)]
if cmc_geom is not None:
    outlines.append(viz.outline_image(cmc_geom, "9467bd", 3))

lst_png = viz.save_thumbnail(
    [recent.select("LST_C").visualize(min=LST_MIN_C, max=LST_MAX_C, palette=THERMAL)]
    + outlines,
    WORK_REGION,
    f"figures/lst_dry_season_{MAP_YEAR}.png",
)
print(f"LST (degC), dry season {MAP_YEAR} - stretch {LST_MIN_C}-{LST_MAX_C} degC")
display(Image(filename=str(lst_png)))

In [ ]:
# COLAB: RUN THIS CELL
obs_png = viz.save_thumbnail(
    [recent.select(params["composites"]["obs_count_band"])
        .visualize(min=0, max=OBS_MAX, palette=COUNTS)]
    + outlines,
    WORK_REGION,
    f"figures/lst_obs_count_{MAP_YEAR}.png",
)
print(f"VALID OBSERVATIONS behind that composite - stretch 0-{OBS_MAX} scenes")
print("Read this together with the map above. CLAUDE.md caveat 2: a warm pixel")
print("backed by one scene is not evidence of anything.")
display(Image(filename=str(obs_png)))

In [ ]:
# COLAB: RUN THIS CELL
# Observation-count statistics over the analysis zone - the honest floor for
# Phase 4 trend fitting.
obs_band = params["composites"]["obs_count_band"]
obs_stats = (
    recent.select(obs_band)
    .reduceRegion(
        reducer=ee.Reducer.min()
        .combine(ee.Reducer.median(), sharedInputs=True)
        .combine(ee.Reducer.max(), sharedInputs=True),
        geometry=zone_geom,
        scale=params["crs"]["analysis_scale_m"],
        maxPixels=params["composites"]["reduce_max_pixels"],
        tileScale=params["composites"]["tile_scale"],
    )
    .getInfo()
)
print(f"Dry-season {MAP_YEAR} valid observations over {zone_label}: {obs_stats}")
print()
print("min == 0 means some pixels had NO usable scene that dry season. That is a")
print("real tropical-cloud result, not a bug - Phase 4 must mask on it, not hide it.")

## Step 5 - MODIS day and night LST

MOD11A2 is a plain average of the daily retrievals with **no built-in quality
filtering**, so `QC_Day` / `QC_Night` are applied explicitly here: mandatory QA
"good quality" (bits 0-1 == 0) **and** average LST error <= 1 K (bits 6-7 == 0).

MODIS is the only night-time source in this project (CLAUDE.md caveat 4). Terra
overpasses ~10:30 / ~22:30 local, Aqua ~13:30 / ~01:30 - Aqua daytime is closest
to peak heating, Terra daytime closest to the Landsat overpass.

Aqua carries no data before **2002-07-04**; `clamp_start_date` warns and clamps
rather than returning empty years that look like data gaps. At 1 km these
collections are cheap compared with Landsat.

In [ ]:
# COLAB: RUN THIS CELL
modis_scale = params["modis_lst"]["reduction_scale_m"]
modis_series = {}

for product in ("terra", "aqua"):
    for daynight in ("day", "night"):
        annual = modis.annual_lst(
            product, daynight, params, geometry=zone_geom, region=WORK_REGION
        )
        modis_series[(product, daynight)] = annual.map(lambda img: img.updateMask(land))

print("MODIS annual composites built:", sorted(modis_series))
print("Overpass times:", params["modis_lst"]["overpass_local_time"])
print()

# One QC sanity check: how much does the filtering actually remove?
_terra_day = modis.lst_collection("terra", "day", params, region=WORK_REGION)
print("Terra day 8-day granules over the study period:", _terra_day.size().getInfo())

## Step 6 - Landsat vs MODIS annual means over the CMC

The two are **not expected to agree in absolute terms**. What matters is whether
they agree in **shape**; the offset is a finding to report, not an error to tune
away. Sources of the offset:

* **30 m vs 1 km.** A 1 km MODIS pixel over central Colombo mixes roofs, roads
  and canopy; the Landsat mean over the same polygon weights them differently.
* **Sampling.** Landsat is one instantaneous ~10:30 overpass per 16 days, on
  clear days only. MODIS is an 8-day average of clear-sky daily retrievals.
* **Overpass time.** Terra day (~10:30) is the fair comparator to Landsat;
  Aqua day (~13:30) sits nearer peak heating and should read warmer.

Four deliberate choices in the cell below, three of them about fitting inside
the Earth Engine memory limit without changing a single number:

* **One reducer for both sides** (`mean`). The project default composites
  Landsat with a median and MODIS with a mean; leaving that mismatch in would
  put a reducer artefact inside a number everyone reads as a sensor difference.
* **`with_percentile=False`.** A percentile reducer has to retain every
  observation per pixel in order to sort them; mean and count are streaming
  accumulators. The comparison plot only uses the mean, so the `p90` band is
  pure cost here. This was the single biggest saving.
* **Batched fetching** (`composites.zonal_batch_years`, default 4). One request
  covering all 26 annual composites has to hold 26 image graphs at once, which
  is what exhausted the memory limit on run 2. Batching changes no numbers, only
  the number of round trips. Drop `BATCH_YEARS` to 1 if it still fails.
* **`ZONAL_SCALE_M = 100`**, not the 30 m analysis grid. A spatial mean over the
  ~47 km2 CMC is insensitive to this. Drop it to 30 if you want the analysis-grid
  number and have the headroom - and if you do, state which scale you used,
  exactly as Phase 1 does for the CMC land area.

The Landsat series is fetched in its own cell so a failure there does not cost
you the four MODIS series (and vice versa).

In [ ]:
# COLAB: RUN THIS CELL
COMPARISON_REDUCER = "mean"
ZONAL_SCALE_M = 100   # see the note above
BATCH_YEARS = 4       # lower to 1 if Earth Engine still reports a memory limit

# with_percentile=False: the plot needs only the mean, and the percentile
# accumulator is the expensive part of a 26-year reduction.
landsat_annual = composites.annual_composites(
    scenes, params, reducer=COMPARISON_REDUCER, with_percentile=False
).map(lambda img: img.updateMask(land))

series = {}
landsat_label = f"Landsat 5/7/8/9 ({COMPARISON_REDUCER}, {ZONAL_SCALE_M} m, ~10:30)"
series[landsat_label] = composites.zonal_annual_means(
    landsat_annual, zone_geom, params, scale_m=ZONAL_SCALE_M, batch_years=BATCH_YEARS
)
print(f"--- {landsat_label} ---")
print(series[landsat_label].to_string(index=False))

In [ ]:
# COLAB: RUN THIS CELL
# MODIS is 1 km, so these are cheap compared with the Landsat series above.
for (product, daynight), collection in sorted(modis_series.items()):
    label = f"MODIS {product.title()} {daynight} (1 km)"
    series[label] = composites.zonal_annual_means(
        collection, zone_geom, params, scale_m=modis_scale, batch_years=BATCH_YEARS
    )
    print(f"--- {label} ---")
    print(series[label].to_string(index=False))
    print()

In [ ]:
# COLAB: RUN THIS CELL
comparison_png = viz.plot_annual_lst_comparison(
    series,
    "figures/lst_landsat_vs_modis_cmc.png",
    params,
    title=f"Annual mean LST over the {zone_label}, {params['time']['start_year']}-"
          f"{params['time']['end_year']}",
)
display(Image(filename=str(comparison_png)))

print("Reminder: these are SURFACE temperatures under clear skies only. A")
print("clear-sky sampling bias warms every series here relative to a true annual")
print("mean, and it is not corrected anywhere in this pipeline.")

## Step 7 - spectral indices spot check

Indices are sensor-agnostic because they read the harmonised `blue`...`swir2`
band names. Albedo uses **one coefficient set across all four sensors** - see
`config/params.yaml`, `indices.albedo`, for why that preserves rather than
breaks continuity at the 2013 sensor change.

In [ ]:
# COLAB: RUN THIS CELL
# Indices on the dry-season scenes of the mapped year only.
_year_scenes = landsat.harmonised_collection(
    params,
    region=zone_geom,
    start_date=f"{MAP_YEAR}-01-01",
    end_date=f"{MAP_YEAR}-04-01",
)
_index_composite = (
    _year_scenes.map(lambda img: indices.add_indices(img, params))
    .median()
    .updateMask(land)
)

_index_bands = list(indices.INDEX_BAND_NAMES.values())
_stats = _index_composite.select(_index_bands).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=zone_geom,
    scale=ZONAL_SCALE_M,
    maxPixels=params["composites"]["reduce_max_pixels"],
    tileScale=params["composites"]["tile_scale"],
).getInfo()

print(f"Mean index values over the {zone_label}, Jan-Mar {MAP_YEAR} (at {ZONAL_SCALE_M} m):")
for _name in _index_bands:
    _value = _stats.get(_name)
    print(f"  {_name:<8}{_value if _value is None else round(_value, 4)}")
print()
print("Plausibility: NDVI ~0.1-0.4 over a dense city core, NDBI positive,")
print("MNDWI negative on land, albedo ~0.10-0.20 for urban fabric.")

## What to check before signing Phase 2 off

1. **Scene inventory** - 2012-05 to 2013-03 is Landsat 7 only; no Landsat 8
   before 2013; no Landsat 9 before late 2021. A year that is unexpectedly zero
   is a bug, not weather.
2. **`L2SP only` counts** are close to the unfiltered counts. A zero there means
   the `PROCESSING_LEVEL` property changed.
3. **Dry-season LST map** - the CMC core reads warmer than its vegetated
   surroundings, water is absent (masked), values roughly 25-45 degC.
4. **Observation-count map** - non-zero across the CMC. Large 0-2 areas are the
   real tropical-cloud floor; record the number, do not hide it.
5. **Landsat vs MODIS** - the series track each other in shape. Aqua day should
   sit warmest, Terra/Aqua night coolest. A constant offset between Landsat and
   Terra day is expected and must be reported as such.
6. **Index means** are physically plausible (see the cell above).

Report back with the printed tables and the three PNGs in `figures/`, and I will
close Phase 2 or fix what they expose.

> Nothing in this notebook has been executed by Claude Code - it has no Earth
> Engine credentials. Until you run it, every Earth Engine cell here is
> unverified.